In [2]:
!pip install langchain-core langchain-community duckduckgo-search langchain_experimental
# !pip install -q langchain langchain-google-genai google-generativeai langchain-text-splitters pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [4]:
os.environ["GOOGLE_API_KEY"]="AQ.Ab8RN6JeexNqt_OqceHinCofM1XteQaV9kCU7clhvBCmy9crzA"
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash')

In [7]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")
documents=loader.load()
report=""
for page in documents:
  report += page.page_content+'\n'

print("Total Pages:",len(documents))
print(report[:1000])

Total Pages: 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)


In [9]:

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks=text_splitter.split_documents(documents)

In [10]:
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

In [11]:
query="What investments did AxiOraa make in Artificial Intelligence"

result=vector_store.similarity_search_with_score(
    query,k=2 #here k is the number of possible outcomes
)

for i, (doc,score) in enumerate(result,start=1):
  print(f"result{i}")
  print(f"similarity_score:{score}")
  print(f"content:{doc.page_content[:1000]}")

result1
similarity_score:0.5444321036338806
content:AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observation

In [12]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

In [13]:
rag_prompt=ChatPromptTemplate.from_template(
    """

    You are a Senior Consultant and your job is to answer the user's questions
    only by using the retrieved context.

    If the answer cannot be found in the context, simply say - "I couldn't find this information"

    Retrieved Context:{context}

    Question:{question}

    Provide:
    1. Answer
    2. Supporting Evidence

    """
)

In [14]:
rag_chain=(
    rag_prompt | llm | StrOutputParser()
)

In [15]:
question="Summarize the AI Investments made by AxiOraa"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke(
    {"context":context,
     "question":question}
)
print(response)

**1. Answer**
AxiOraa Ltd. made significant AI investments focused on core infrastructure, operational tools, and generative AI capabilities. Specifically, the company invested in:
* LLMOps
* Vector databases
* Evaluation frameworks
* LangChain accelerators
* AI observability

**2. Supporting Evidence**
* *"Significant investments were made in LLMOps, vector databases, evaluation frameworks, LangChain accelerators and AI observability."*


In [16]:

question="In bullet points share the highlights from the CEO's message"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke(
    {"context":context,
     "question":question}
)
print(response)


**1. Answer**

Here are the key highlights from the CEO's Message for FY2025:

* **Revenue Growth:** FY2025 experienced strong revenue growth, increasing by 21.1% from USD 2.18B to USD 2.64B, primarily driven by enterprise AI adoption.
* **Operational Efficiency:** Investments in proprietary AI accelerators and strategic partnerships enhanced delivery efficiency, helping navigate macroeconomic uncertainty.
* **Margin Expansion:** 
  * Gross Margin improved from 41.5% to 44.2%.
  * EBITDA Margin reached 22.8%.
* **Profitability & Cash Flow:** Net Profit rose to USD 352M, while Free Cash Flow remained strong.

---

**2. Supporting Evidence**

The above highlights are directly supported by the following quotes from the retrieved context:

* *"FY2025 was marked by strong revenue growth driven by enterprise AI adoption."*
* *"Investments in proprietary AI accelerators and strategic partnerships improved delivery efficiency despite macroeconomic uncertainty."*
* *"Revenue grew from USD 2.18B